In [0]:
%python
bronze_path = '/Volumes/workspace/techvenda/filestore/bronze/'
silver_path = '/Volumes/workspace/techvenda/filestore/silver/'
gold_path = '/Volumes/workspace/techvenda/filestore/gold/'
origem_path = '/Volumes/workspace/techvenda/filestore/origem/'

In [0]:
%python
#Tabelas temporarias
bronze_mapeamento= {
    'temp_bronze_clientes' : f'{bronze_path}/clientes/',
    'temp_bronze_itens_pedido' : f'{bronze_path}/itens_pedido/',
    'temp_bronze_pedidos' : f'{bronze_path}/pedidos/',
    'temp_bronze_produtos' : f'{bronze_path}/produtos/',
    'temp_bronze_vendedores' : f'{bronze_path}/vendedores/'

}
for view_name, path in bronze_mapeamento.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)

)

In [0]:
%sql
select * from temp_bronze_itens_pedido

In [0]:
%sql
describe temp_bronze_itens_pedido

In [0]:
%python
df_itens_pedido = spark.sql("""
    SELECT
        id_item,
        id_pedido,
        id_produto,
        quantidade,
        marca

    FROM temp_bronze_itens_pedido
    GROUP BY
        id_item,
        id_pedido,
        id_produto,
        quantidade,
        marca
""")


# Salvar em delta na silver
df_itens_pedido.write\
    .mode('overwrite')\
        .format('delta')\
            .option('mergeSchema', 'true')\
                .save(f'{silver_path}/itens_pedido')

In [0]:
%sql
select * from temp_bronze_itens_pedido